# DocQWise — Getting Started Notebook

## 📄 Read. Extract. Retrieve.

**Document intelligence that adapts, accelerates, and scales.**

This notebook walks you through every core feature of docqwise.

### Install
```bash
pip install docqwise
```

---

## 1. Setup

In [ ]:
from docqwise import Docqwise, __version__

print(f"DocQWise v{__version__}")
dq = Docqwise()
print(dq)
print(dq.metrics())

## 2. Read a PDF

In [ ]:
from docqwise.readers.pdf_reader import PDFReader

reader = PDFReader()
doc = reader.read("demo/sample_invoice.pdf")

print(f"Pages: {doc.metadata.page_count}")
print(f"Words: {doc.metadata.word_count}")
print(f"Scanned: {doc.metadata.is_scanned}")
print(f"Hash: {doc.metadata.file_hash}")
print(f"\nText:\n{doc.text[:500]}")

## 3. Auto Format Detection

DocQWise auto-detects 36 file formats and 5 source types.

In [ ]:
from docqwise.readers.auto_reader import AutoReader

auto = AutoReader()

files = ["report.pdf", "contract.docx", "scan.tiff", "data.xlsx", 
         "records.csv", "config.json", "slides.pptx", "mail.eml",
         "postgresql://host/db", "s3://bucket/docs/"]

for f in files:
    dt = auto.detect_type(f)
    st = auto.detect_source_type(f)
    print(f"{f:35s} format={dt.value:10s} source={st.value}")

print(f"\nTotal supported extensions: {len(auto.supported_formats())}")

## 4. Field Extraction Result

Every field has a typed value, confidence score, bounding box location, and extraction method.

In [ ]:
from docqwise.core.field import FieldResult, ExtractionResult
from docqwise.core.chunk import BoundingBox

result = ExtractionResult(
    doc_id="inv_001",
    source_path="demo/sample_invoice.pdf",
    fields={
        "vendor_name": FieldResult(value="Acme Corporation", confidence=0.97,
            bbox=BoundingBox(50, 190, 200, 205, page=1), page=1, extraction_method="text_match"),
        "invoice_number": FieldResult(value="INV-2026-0042", confidence=0.99,
            page=1, extraction_method="regex"),
        "total": FieldResult(value=218300.00, confidence=0.98,
            page=1, extraction_method="table_sum"),
        "tax": FieldResult(value=33300.00, confidence=0.95,
            page=1, extraction_method="text_match"),
        "due_date": FieldResult(value="2026-09-01", confidence=0.96,
            page=1, extraction_method="date_parser"),
    },
    confidence=0.97,
    strategy_used="fast",
    processing_time_ms=45.2,
)

for name, field in result.fields.items():
    print(f"{name:20s} = {str(field.value):20s} confidence={field.confidence:.2f}")

print(f"\nJSON output:\n{result.to_json()}")

## 5. Self-Improving Corrections

Fix a wrong field. Stored as exact override. Applied automatically next time.

No ML. No retraining. Deterministic.

In [ ]:
print("BEFORE:")
print(f"  tax = {result.fields['tax'].value} (corrected: {result.fields['tax'].is_corrected})")

result.correct({"tax": 35000.00, "po_number": "PO-2026-8891"})

print("\nAFTER:")
print(f"  tax = {result.fields['tax'].value} (corrected: {result.fields['tax'].is_corrected})")
print(f"  po_number = {result.fields['po_number'].value} (new field added!)")

## 6. Correction Store

Corrections persist in SQLite. Applied automatically on future documents.

In [ ]:
from docqwise.learning.correction import CorrectionStore
import shutil

store = CorrectionStore("/tmp/docqwise_notebook")

store.save("inv_001", "invoice_acme_001.pdf", {"tax": 35000.00, "po_number": "PO-8891"})
store.save("inv_002", "invoice_acme_002.pdf", {"department": "Engineering"})

print(f"Corrections stored: {store.count()}")

for c in store.list_corrections():
    print(f"  {c['source_path']}: {c['field_name']} = {c['correct_value']}")

# Apply to new document
original = {"tax": 0.0, "total": 218300.0}
corrected = store.apply("invoice", original)
print(f"\nOriginal tax: {original['tax']} -> Corrected: {corrected['tax']}")

shutil.rmtree("/tmp/docqwise_notebook")

## 7. Table Extraction

Tables extracted with full structure. Export to DataFrame, CSV, JSON, Markdown, Excel.

In [ ]:
from docqwise.core.element import Table, TableCell

cells_data = [
    TableCell(text="Product", row=0, col=0, is_header=True),
    TableCell(text="Qty", row=0, col=1, is_header=True),
    TableCell(text="Price", row=0, col=2, is_header=True),
    TableCell(text="Total", row=0, col=3, is_header=True),
    TableCell(text="Widget A", row=1, col=0),
    TableCell(text="100", row=1, col=1),
    TableCell(text="50.00", row=1, col=2),
    TableCell(text="5000.00", row=1, col=3),
    TableCell(text="Widget B", row=2, col=0),
    TableCell(text="200", row=2, col=1),
    TableCell(text="35.00", row=2, col=2),
    TableCell(text="7000.00", row=2, col=3),
    TableCell(text="Widget C", row=3, col=0),
    TableCell(text="75", row=3, col=1),
    TableCell(text="120.00", row=3, col=2),
    TableCell(text="9000.00", row=3, col=3),
]

table = Table(table_id="tbl_001", cells=cells_data, rows=4, cols=4,
              page=1, caption="Quarterly Sales", confidence=0.96)

# DataFrame
df = table.to_dataframe()
print("DataFrame:")
print(df)

# Markdown
print("\nMarkdown:")
print(table.to_markdown())

# JSON
import json
print("\nJSON:")
print(json.dumps(table.to_json(), indent=2))

## 8. Entity and Relation Extraction

In [ ]:
from docqwise.core.element import Entity, Relation

entities = [
    Entity(text="Acme Corporation", entity_type="ORG", confidence=0.98),
    Entity(text="TechStart Solutions", entity_type="ORG", confidence=0.97),
    Entity(text="INR 2,18,300", entity_type="MONEY", confidence=0.99),
    Entity(text="August 1, 2026", entity_type="DATE", confidence=0.96),
]

print("Entities:")
for e in entities:
    print(f"  [{e.entity_type:8s}] {e.text}")

relations = [
    Relation("Acme Corp", "provides_service_to", "TechStart Solutions"),
    Relation("TechStart", "shall_pay", "INR 2,18,300"),
    Relation("Agreement", "governed_by", "laws of India"),
]

print("\nRelations:")
for r in relations:
    print(f"  {r.subject} --[{r.predicate}]--> {r.object}")

## 9. Configuration

YAML or JSON config. Everything customizable.

In [ ]:
from docqwise.config import DocqwiseConfig, StrategyConfig, SpeedConfig

# Default
config = DocqwiseConfig()
print(f"Strategy:  {config.strategy.mode}")
print(f"Workers:   {config.speed.workers}")
print(f"Store:     {config.storage.vector_store}")
print(f"Search:    {config.search.mode}")
print(f"Chunker:   {config.chunker.strategy}")

# Production
prod = DocqwiseConfig(
    strategy=StrategyConfig(mode="hybrid", confidence_threshold=0.90),
    speed=SpeedConfig(workers=8, gpu_workers=2, batch_size=64),
)
print(f"\nProduction: {prod.strategy.mode}, {prod.speed.workers} workers, {prod.speed.gpu_workers} GPUs")

## 10. Different Engine Configurations

Same API. Different backends. User picks their stack.

In [ ]:
# Default — zero dependencies
print(Docqwise())

# Production — parallel processing
print(Docqwise(strategy="hybrid", workers=8, gpu_workers=2))

# Local-only — air-gapped
print(Docqwise(strategy="local_only", workers=4))

# Custom vector store
print(Docqwise(vector_store="qdrant", search_mode="hybrid"))

# API-first — max accuracy
print(Docqwise(strategy="api_first", ocr="azure_doc_intelligence"))

## 11. Pluggable Base Classes

Every component is swappable. 18 abstract base classes.

In [ ]:
from docqwise.core import bases
import inspect

base_classes = [(name, obj) for name, obj in inspect.getmembers(bases) 
                if inspect.isclass(obj) and name.startswith("Base")]

print(f"Pluggable interfaces: {len(base_classes)}")
for name, cls in base_classes:
    methods = [m for m in dir(cls) if not m.startswith("_")]
    print(f"  {name:22s} methods: {', '.join(methods)}")

---

## What's Next

The full extraction pipeline, RAG engine, GraphRAG, agentic pipelines, MCP server, and ecosystem bridges are coming next.

## Ecosystem

| Library | Tagline | Domain |
|---|---|---|
| SightRAG | See. Search. Retrieve. | Visual intelligence |
| sonarwise | Hear. Search. Retrieve. | Audio intelligence |
| **docqwise** | **Read. Extract. Retrieve.** | **Document intelligence** |
| adaptive-intelligence | Learn. Remember. Adapt. | Orchestration |
| llmevalkit | Evaluate. Score. Improve. | Evaluation |

---

**DocQWise: Read. Extract. Retrieve.**

*Sometimes the ant carries the elephant.* 🐜